<a href="https://colab.research.google.com/github/DzeGra/reducto-financial-extraction/blob/main/apple_q3_fy2024_10q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reducto Financial Extraction — Apple Q3 FY2024 Form 10-Q
**Use case:** an LP monitoring portfolio companies and public comparables needs quarterly filing data in a standard Excel format. Today analysts key it in by hand.

**Pipeline:** PDF → Reducto upload → Parse → Extract (JSON schema, citations on) → verification → validation → LP workbook

Same pipeline as the Amazon notebook: the shared helpers live in `src/build_workbook.py`; Apple adds its own schema and tabs (`src/build_apple_workbook.py`).

| Step | What happens |
|---|---|
| 1 | Setup and authentication |
| 2 | Upload the 10-Q |
| 3 | Load the 207-field schema |
| 4 | Parse (layout, tables, citations) |
| 5 | Extract every field |
| 6 | Inspect one field and its citation |
| 7 | Verify against an answer key and the PDF |
| 8 | Build and download the LP workbook |

## 1. Setup and authentication
The API key lives in Colab Secrets (🔑 icon in the left sidebar) under the name `REDUCTO_API_KEY`, so it never appears in the notebook.

In [ ]:
!pip install -q reductoai openpyxl pdfplumber
!git clone -q https://github.com/DzeGra/reducto-financial-extraction.git 2>/dev/null || git -C reducto-financial-extraction pull -q

import sys, json, requests
sys.path.append("reducto-financial-extraction/src")
REPO = "reducto-financial-extraction"

from pathlib import Path
from google.colab import userdata, files
from reducto import Reducto
from build_apple_workbook import build_apple_workbook
import verify_against_pdf as verify

client = Reducto(api_key=userdata.get("REDUCTO_API_KEY"))
print("Reducto client ready")

## 2. Upload the 10-Q
Select `_10-Q-Q3-2024-As-Filed.pdf` when prompted. Reducto returns a file reference that parse and extract both use.

In [ ]:
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]

upload = client.upload(file=Path(PDF_PATH))
print("Uploaded:", PDF_PATH)

## 3. Load the extraction schema
The schema is the contract with the LP: one field per number they asked for, each describing the exact table, period column, unit and sign. It covers the income statement, gross margin by Products and Services, net sales by category, geographic segments, cash flow, balance sheet, capital return and the qualitative disclosures.
`schema/generate_apple_schema.py` writes it, so the four-period repeats are never typed by hand.

In [ ]:
schema = json.load(open(f"{REPO}/schema/apple_q3_fy2024_10q_schema.json"))["schema"]
print(len(schema["properties"]), "fields in the schema")
print(json.dumps(schema["properties"]["q3_fy2024_total_net_sales"], indent=2))

## 4. Parse
The same settings used in Reducto Studio: model r-1, citations, page markers and HTML tables (which keep the multi-level Three Months / Nine Months headers intact).

In [ ]:
parse_result = client.parse.run(
    input=upload,
    formatting={"add_page_markers": True, "table_output_format": "html"},
    settings={"timeout": 900, "model": "r-1", "return_citations": True},
)
print("Parse job:", parse_result.job_id)
print("Pages:", parse_result.usage.num_pages)

## 5. Extract
The code comes from Studio's **Code** tab, with the schema loaded from the repo file instead of pasted inline. Citations (V2 model) return the page, bounding box and confidence for every value.
Set `REUSE_JOB_ID` to a completed job ID to skip re-running and save credits.

In [ ]:
REUSE_JOB_ID = "92b16dab-a0ce-4115-91e5-a03f2a6f2c62"   # demo day job, saved with persist_results

SYSTEM_PROMPT = (
    "You are extracting financial data from Apple Inc.'s Form 10-Q for the fiscal quarter "
    "ended June 29, 2024. Follow each field description exactly: use the named table and "
    "period column. Amounts are in USD millions unless the field says otherwise. "
    "Return null when the value is not stated in the document."
)

if REUSE_JOB_ID:
    response, job_id = client.job.get(REUSE_JOB_ID), REUSE_JOB_ID
else:
    response = client.extract.run(
        input=upload,
        instructions={"schema": schema, "system_prompt": SYSTEM_PROMPT},
        parsing={"settings": {"persist_results": True}},
        settings={"citations": {"enabled": True, "numerical_confidence": False},
                  "alpha": {"citation_model": "v2"}},
    )
    job_id = response.job_id

def find_fields(data):
    """Walk the response (inline result, nested job result or download URL) to the extracted fields."""
    data = data.model_dump() if hasattr(data, "model_dump") else data
    for _ in range(6):
        if isinstance(data, list) and len(data) == 1:
            data = data[0]
        if isinstance(data, dict) and "q3_fy2024_total_net_sales" in data:
            return data
        if isinstance(data, dict) and isinstance(data.get("url"), str):
            data = requests.get(data["url"], timeout=120).json()
        elif isinstance(data, dict) and "result" in data:
            data = data["result"]
    raise ValueError("Could not find the extracted fields in the Reducto response")

fields = find_fields(response)
print("Extract job:", job_id)
print("Fields returned:", len(fields))
print("Total net sales Q3 FY24:", fields["q3_fy2024_total_net_sales"]["value"])

## 6. Inspect one field and its citation
Every value carries the text it came from, the page, its position and a confidence score: the same bounding box Reducto Studio highlights on the PDF.

In [ ]:
item = fields["ytd_fy2024_commercial_paper_net"]
cite = item["citations"][0]

print("Value:     ", item["value"])
print("Page:      ", cite["bbox"]["page"])
print("Position:  ", {k: round(cite["bbox"][k], 3) for k in ("left", "top", "width", "height")})
print("Confidence:", cite.get("granular_confidence"))
print("Source:    ", cite.get("content"))

## 7. Verify
Two independent checks:
1. **Answer key:** every value compared to numbers read by hand from the filing (`tests/apple_q3_fy2024_expected.json`).
2. **PDF page check:** each number must be printed on the page Reducto cited. `pdfplumber` reads the raw page text; it only answers "is this number on this page?"

In [ ]:
expected = json.load(open(f"{REPO}/tests/apple_q3_fy2024_expected.json"))["values"]
values = {name: (f.get("value") if isinstance(f, dict) else f) for name, f in fields.items()}

rows = verify.compare_to_expected(values, expected)
matches = sum(1 for _, _, _, ok in rows if ok)
print(f"Answer key: {matches} of {len(rows)} match")
for field, got, exp, ok in rows:
    if not ok:
        print("  MISMATCH:", field, "| Reducto:", got, "| answer key:", exp)

pdf_rows = verify.verify_payload(fields, PDF_PATH)
summary = verify.summary(pdf_rows)
print("PDF page check:", summary)
for r in pdf_rows:
    if r["status"] != "ON PAGE":
        print(" ", r["status"], "|", r["field"], "| cited page:", r["page"])

## 8. Build the LP workbook
Tabs match the LP brief: **Financials, Segments, Cash Flow, Guidance, Notes**, plus **Validation** (28 reconciliation checks: segments and categories add up to totals, the income statement math holds, cash flows reconcile to the change in cash, and the balance sheet balances).
Growth rates and margins are Excel formulas, so an analyst can click any cell and see the math.

In [ ]:
path, checks = build_apple_workbook(
    fields,
    "Apple_Q3_FY2024_LP_Analysis.xlsx",
    meta={"job_id": job_id, "n_fields": len(schema["properties"]),
          "answer_key_matches": matches, "answer_key_total": len(rows),
          "on_page": summary.get("ON PAGE", 0), "nulls": summary.get("NULL", 0)},
)
passed = sum(abs(calc - reported) < 0.5 for _, calc, reported in checks)
print(f"Validation: {passed} of {len(checks)} checks passed")
files.download(path)

## Results
- **Answer key:** 201 of 201 fields match, including 2 correct nulls (2024 term debt proceeds is a dash; Apple gives no quantitative guidance in a 10-Q).
- **Traceability:** 199 of 199 numbers are printed on the exact page Reducto cited.
- **Reconciliation:** 28 of 28 validation checks pass.
- **Reuse:** the Apple workbook shares its helpers with the Amazon builder; only the schema and tabs are company-specific.